In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts import lightning_ssl
from jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals

sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path

from lightning_scripts.eval_jsin_transfer import SSLWordClassifier

In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/pilot_ssl_barlow_dualtask_resnet50_hparam_set_1_lr_02_LARS_MatchedSpeechInNoiseDatasetBatched.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

TASK = "speaker"
LAYER = "avgpool" ## ckpt at this layerneeds to exist 

config['data'] = {}
config['data']['root'] = "/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/"
config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['data']['eval_max'] = 3
# config['hparas']['optimizer'] = args.optimizer
# config['hparas']['lr'] = args.lr * args.gpus
# config['hparas']['epochs'] = 2
# don't load in classifier head if it exists 
config['model']['arch_kwargs']['supervised'] =  False

if TASK == "word":
    config['data']['task_label'] = 'signal/word_int'

elif TASK == "speaker":
    config['data']['task_label'] = 'signal/speaker_int'
    config['model']['arch_kwargs']['n_classes'] =  433

In [10]:
model_ckpt_dir = Path(f"model_checkpoints/{config_path.stem}/")
task_ckpt_dirs = list(model_ckpt_dir.glob(f"linear_classifier_checkpoints_{TASK}*best_val*"))
ckpt_dir = task_ckpt_dirs[0]

ckpt_path = list(ckpt_dir.rglob("*.ckpt"))[0]
# dummy init with checkpoint 
module = SSLWordClassifier(config=config, ckpt_path=ckpt_path, layer_out=LAYER)


checkpoint = torch.load(ckpt_path, weights_only=True)

module.load_state_dict(checkpoint['state_dict'])
## update keys to remove _orig_mod from eatch key 

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:191: Found keys that are in the model state dict but not in the checkpoint: ['audio_rep.rep.downsampling_op.downsample_filter', 'audio_rep.rep.Cochleagram.compute_subbands.coch_filters', 'audio_rep.rep.Cochleagram.downsampling.downsample_filter', 'model.front_end.rep.downsampling_op.downsample_filter', 'model.front_end.rep.Cochleagram.compute_subbands.coch_filters', 'model.front_end.rep.Cochleagram.downsampling.downsample_filter', 'model.model.f.conv1.weight', 'model.model.f.bn1.weight', 'model.model.f.bn1.bias', 'model.model.f.bn1.running_mean', 'model.model.f.bn1.running_var', 'model.model.f.layer1.0.conv1.weight', 'model.model.f.layer1.0.bn1.weight', 'model.model.f.layer1.0.bn1.bias', 'model.model.f.layer1.0.bn1.running_mean', 'model.model.f.layer1.0.bn1.running_var', 'model.model.f.layer1.0.conv2.weight', 'model.model.f.layer1.0.bn2.weight', 'model.model.f.layer1.0.bn2.bias', 'mode

<All keys matched successfully>

In [4]:
trainer = L.Trainer(devices=1)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable Tenso

In [5]:
# run test 
test_dataset = jsinV3_precombined_all_signals(root=config['data']['root'],
                                                train=False,
                                                transform=None,
                                                batch_size=config['hparas']['batch_size'],
                                                eval_max=1)
test_dataset.target_keys = ['signal/word_int']
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=module.collate_fn
)


In [6]:
outputs = trainer.predict(module, test_dataloader, return_predictions=True)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [9]:
output_vals = torch.cat([output['accuracy'] for output in outputs])
output_vals.mean()

tensor(0.7014)